In [22]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_curve,
    auc,
    confusion_matrix,
)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from matplotlib import rcParams
from tqdm import tqdm
import shap
import time

In [23]:
# ===================== DEVICE SETUP ===================== #
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [24]:
# ===================== PATHS AND CONFIGURATION ===================== #
model_name = "convnext"
OUTPUT_PATH = f"1_Feature_Extraction/{model_name}"
RESULTS_PATH = os.path.join("7_Epoch_Count_Comparison")
os.makedirs(RESULTS_PATH, exist_ok=True)

# Load feature data
try:
    train_df = pd.read_csv(os.path.join(OUTPUT_PATH, f"train_features_{model_name}.csv"))
    test_df = pd.read_csv(os.path.join(OUTPUT_PATH, f"test_features_{model_name}.csv"))
    print(f"Loaded {len(train_df)} training and {len(test_df)} testing samples")
except FileNotFoundError:
    print(f"ConvNext features not found, attempting to use available features")
    train_df = pd.read_csv(os.path.join(f"1_Feature_Extraction/{model_name}", f"train_features_{model_name}.csv"))
    test_df = pd.read_csv(os.path.join(f"1_Feature_Extraction/{model_name}", f"test_features_{model_name}.csv"))
    print(f"Using {model_name} features with {len(train_df)} training and {len(test_df)} testing samples")

Loaded 5712 training and 1311 testing samples


In [25]:
# Extract features and labels
feature_columns = [col for col in train_df.columns if col.startswith("feat_")]
X_train = train_df[feature_columns].values
y_train = train_df["label"].map({"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}).values
X_test = test_df[feature_columns].values
y_test = test_df["label"].map({"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}).values

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
N_CLASSES = len(CLASSES)  # Should be 4
print(f"Number of classes: {N_CLASSES}")

Number of classes: 4


In [26]:
# Plot settings
rcParams["font.family"] = "Times New Roman"
rcParams["axes.titlesize"] = 28
rcParams["axes.titlepad"] = 20
rcParams["axes.labelsize"] = 23
rcParams["xtick.labelsize"] = 18
rcParams["ytick.labelsize"] = 18
rcParams["legend.fontsize"] = 16
rcParams["lines.linewidth"] = 3
rcParams["axes.linewidth"] = 2

In [27]:
# ===================== FEATURE SELECTION USING SHAP ===================== #
def apply_shap(X_train, X_test, n_features):
    print(f"Applying SHAP to select {n_features} features...")
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
    rf_model.fit(X_train, y_train)
    feature_importance = rf_model.feature_importances_
    selected_indices = np.argsort(feature_importance)[::-1][:n_features]
    selected_indices = np.array(selected_indices, dtype=int)
    X_train_selected = X_train[:, selected_indices]
    X_test_selected = X_test[:, selected_indices]
    return X_train_selected, X_test_selected, selected_indices

In [28]:
# ===================== MODEL DEFINITION ===================== #
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [29]:
class Attention(nn.Module):
    def __init__(self, feature_dim):
        super(Attention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1),
            nn.Softmax(dim=1),
        )

    def forward(self, x):
        weights = self.attention(x)
        return (x * weights).sum(dim=1)

In [30]:
class AttGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(AttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

In [31]:
def train_model(model, train_loader, test_loader, feature_method, 
               learning_rate=0.001, num_epochs=50, weight_decay=1e-5, 
               max_epochs=None, early_stopping=False):
    criterion = nn.CrossEntropyLoss()
    
    # Initialize AdamW optimizer with fixed learning rate
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    train_losses, test_losses, train_accs, test_accs = [], [], [], []
    
    # For measuring training time
    start_time = time.time()
    
    # Early stopping parameters
    best_test_acc = 0
    patience = 20  # Number of epochs to wait for improvement
    patience_counter = 0
    
    # Determine the actual number of epochs to train
    actual_epochs = min(num_epochs, max_epochs) if max_epochs else num_epochs

    for epoch in tqdm(range(actual_epochs), desc=f"Training AttGRU for {actual_epochs} epochs with AdamW LR={learning_rate}"):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        test_loss = test_loss / len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        # Early stopping check
        if early_stopping:
            if test_acc > best_test_acc:
                best_test_acc = test_acc
                patience_counter = 0
            else:
                patience_counter += 1
                
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")
    
    return train_losses, test_losses, train_accs, test_accs, training_time

In [32]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        outputs = model(X_test_tensor)
        _, y_pred = outputs.max(1)
        y_pred = y_pred.cpu().numpy()
        y_prob = torch.softmax(outputs, dim=1).cpu().numpy()

    metrics = {}
    metrics["ACC"] = accuracy_score(y_test, y_pred)
    metrics["AUC"] = roc_auc_score(y_test, y_prob, multi_class="ovr")
    metrics["PRE"] = precision_score(y_test, y_pred, average="macro")
    metrics["SN"] = recall_score(y_test, y_pred, average="macro")
    
    # Calculate specificity for multiclass
    cm = confusion_matrix(y_test, y_pred)
    specificity_scores = []
    for i in range(N_CLASSES):
        # True negatives are all elements of the confusion matrix except for the current class
        tn = np.sum(cm) - np.sum(cm[i, :]) - np.sum(cm[:, i]) + cm[i, i]
        fp = np.sum(cm[:, i]) - cm[i, i]
        # Avoid division by zero
        if (tn + fp) == 0:
            specificity_scores.append(0)
        else:
            specificity_scores.append(tn / (tn + fp))
    metrics["SP"] = np.mean(specificity_scores)
    
    metrics["F1"] = f1_score(y_test, y_pred, average="macro")
    metrics["MCC"] = matthews_corrcoef(y_test, y_pred)
    return metrics, y_prob, y_pred

In [33]:
def plot_roc(y_test, y_prob, epoch_count):
    y_test_bin = label_binarize(y_test, classes=range(N_CLASSES))  # 4 classes
    fpr, tpr, roc_auc = {}, {}, {}
    plt.figure(figsize=(8, 8))

    for i in range(N_CLASSES):  # Loop over 4 classes
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f"{CLASSES[i]}")

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"AttGRU with {epoch_count} Epochs - ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(True)
    
    # Save as PNG
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_AdamW_Epochs_{epoch_count}_roc_curve.png"), dpi=1000, bbox_inches="tight")
    # Save as PDF
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_AdamW_Epochs_{epoch_count}_roc_curve.pdf"), format='pdf', bbox_inches="tight")
    plt.close()

In [34]:
def plot_learning_curves(train_losses, test_losses, train_accs, test_accs, epoch_count):
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title(f'AdamW {epoch_count} Epochs - Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(test_accs, label='Test Accuracy')
    plt.title(f'AdamW {epoch_count} Epochs - Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    # Save as PNG
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_AdamW_Epochs_{epoch_count}_learning_curves.png"), dpi=1000, bbox_inches="tight")
    # Save as PDF
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_AdamW_Epochs_{epoch_count}_learning_curves.pdf"), format='pdf', bbox_inches="tight")
    plt.close()

In [35]:
print("Starting feature selection and AdamW epoch comparison...")

# Define a fixed number of features to use
n_features = 800
print(f"\n===== Selecting {n_features} features using SHAP =====")
X_train_shap, X_test_shap, shap_indices = apply_shap(X_train, X_test, n_features=n_features)

Starting feature selection and AdamW epoch comparison...

===== Selecting 800 features using SHAP =====
Applying SHAP to select 800 features...


In [36]:
# Define epoch counts to compare
epoch_counts = [50, 100, 200, 300, 400, 500]
epoch_results = []

In [37]:
# Fixed learning rate for AdamW
learning_rate = 0.001

# Prepare datasets and dataloaders
batch_size = 64
train_dataset = FeatureDataset(X_train_shap, y_train)
test_dataset = FeatureDataset(X_test_shap, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [38]:
for epochs in epoch_counts:
    print(f"\n===== Evaluating with {epochs} epochs =====")
    
    # Initialize a new model for each epoch count
    model = AttGRU(input_dim=X_train_shap.shape[1]).to(device)
    
    # Train with the current epoch count using AdamW
    train_losses, test_losses, train_accs, test_accs, training_time = train_model(
        model, 
        train_loader, 
        test_loader, 
        feature_method=f"SHAP_{n_features}", 
        learning_rate=learning_rate,
        num_epochs=epochs  # Set number of epochs
    )

    # Plot learning curves
    plot_learning_curves(train_losses, test_losses, train_accs, test_accs, epochs)

    # Evaluate the model
    metrics, y_prob, y_pred = evaluate_model(model, X_test_shap, y_test)
    
    # Plot ROC curve
    plot_roc(y_test, y_prob, epochs)
    
    # Save model
    torch.save(model.state_dict(), os.path.join(RESULTS_PATH, f"AttGRU_AdamW_Epochs_{epochs}_model.pth"))

    # Save results
    epoch_results.append(
        {
            "Epoch_Count": epochs,
            "Training_Time": training_time,
            "ACC": metrics["ACC"],
            "AUC": metrics["AUC"],
            "PRE": metrics["PRE"],
            "SN": metrics["SN"],
            "SP": metrics["SP"],
            "F1": metrics["F1"],
            "MCC": metrics["MCC"],
        }
    )
    print(f"AttGRU with {epochs} epochs - Performance Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    print(f"Training Time: {training_time:.2f} seconds")


===== Evaluating with 50 epochs =====


Training AttGRU for 50 epochs with AdamW LR=0.001: 100%|██████████| 50/50 [00:36<00:00,  1.36it/s]


Training completed in 36.84 seconds
AttGRU with 50 epochs - Performance Metrics:
ACC: 0.9008
AUC: 0.9842
PRE: 0.8994
SN: 0.8929
SP: 0.9671
F1: 0.8947
MCC: 0.8676
Training Time: 36.84 seconds

===== Evaluating with 100 epochs =====


Training AttGRU for 100 epochs with AdamW LR=0.001: 100%|██████████| 100/100 [01:25<00:00,  1.17it/s]


Training completed in 85.42 seconds
AttGRU with 100 epochs - Performance Metrics:
ACC: 0.9077
AUC: 0.9855
PRE: 0.9039
SN: 0.9004
SP: 0.9694
F1: 0.9017
MCC: 0.8763
Training Time: 85.42 seconds

===== Evaluating with 200 epochs =====


...

Training completed in 199.33 seconds
AttGRU with 300 epochs - Performance Metrics:
ACC: 0.9085
AUC: 0.9844
PRE: 0.9046
SN: 0.9012
SP: 0.9697
F1: 0.9025
MCC: 0.8773
Training Time: 199.33 seconds

===== Evaluating with 400 epochs =====


Training AttGRU for 400 epochs with AdamW LR=0.001: 100%|██████████| 400/400 [03:13<00:00,  2.07it/s]


Training completed in 193.45 seconds
AttGRU with 400 epochs - Performance Metrics:
ACC: 0.9069
AUC: 0.9847
PRE: 0.9037
SN: 0.8996
SP: 0.9691
F1: 0.9011
MCC: 0.8753
Training Time: 193.45 seconds

===== Evaluating with 500 epochs =====


Training AttGRU for 500 epochs with AdamW LR=0.001: 100%|██████████| 500/500 [04:04<00:00,  2.04it/s]


Training completed in 244.96 seconds
AttGRU with 500 epochs - Performance Metrics:
ACC: 0.9100
AUC: 0.9855
PRE: 0.9068
SN: 0.9029
SP: 0.9701
F1: 0.9043
MCC: 0.8794
Training Time: 244.96 seconds


In [39]:
# Save comparison results to CSV
epoch_df = pd.DataFrame(epoch_results)
epoch_df.to_csv(os.path.join(RESULTS_PATH, "AttGRU_AdamW_epoch_count_comparison.csv"), index=False)
print(f"Saved epoch count comparison results to {os.path.join(RESULTS_PATH, 'AttGRU_AdamW_epoch_count_comparison.csv')}")

Saved epoch count comparison results to 7_Epoch_Count_Comparison/AttGRU_AdamW_epoch_count_comparison.csv


In [40]:
# Plot comparison results for performance metrics
plt.figure(figsize=(14, 8))
metrics_to_plot = ["ACC", "AUC", "SP", "SN", "F1", "MCC"]
x = np.arange(len(epoch_counts))
width = 0.15

for i, metric in enumerate(metrics_to_plot):
    values = [result[metric] for result in epoch_results]
    plt.bar(x + i * width, values, width, label=metric)

plt.xlabel("Number of Epochs")
plt.ylabel("Score")
plt.title("AttGRU Performance with AdamW at Different Epoch Counts")
plt.xticks(x + width * (len(metrics_to_plot) - 1) / 2, [str(epochs) for epochs in epoch_counts])
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout()

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_epoch_count_comparison.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_epoch_count_comparison.pdf"), dpi=1000, format='pdf', bbox_inches="tight")
plt.close()

In [41]:
# Plot training time comparison
plt.figure(figsize=(10, 6))
training_times = [result["Training_Time"] for result in epoch_results]
plt.plot(epoch_counts, training_times, 'o-', linewidth=2, markersize=10)
plt.grid(True)
plt.xlabel("Number of Epochs")
plt.ylabel("Training Time (seconds)")
plt.title("Training Time vs Epoch Count")

# Add data labels
for i, time_value in enumerate(training_times):
    plt.annotate(f"{time_value:.1f}s", 
                 (epoch_counts[i], time_value),
                 textcoords="offset points",
                 xytext=(0,10), 
                 ha='center')

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_training_time_comparison.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_training_time_comparison.pdf"), dpi=1000, format='pdf', bbox_inches="tight")
plt.close()

In [42]:
# Plot metrics vs training time
plt.figure(figsize=(14, 10))
for i, metric in enumerate(metrics_to_plot):
    plt.subplot(2, 3, i+1)
    metric_values = [result[metric] for result in epoch_results]
    plt.plot(training_times, metric_values, 'o-', linewidth=2, markersize=8)
    
    # Add epoch count annotations
    for j, epoch_count in enumerate(epoch_counts):
        plt.annotate(f"{epoch_count}", 
                     (training_times[j], metric_values[j]),
                     textcoords="offset points",
                     xytext=(0,7), 
                     ha='center',
                     fontsize=9)
    
    plt.grid(True)
    plt.xlabel("Training Time (seconds)")
    plt.ylabel(metric)
    plt.title(f"{metric} vs Training Time")

plt.tight_layout()

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_metrics_vs_time.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_metrics_vs_time.pdf"), dpi=1000, format='pdf', bbox_inches="tight")
plt.close()